In [13]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [8]:
# 1. simulate more data + augmentation
np.random.seed(42)

# base patterns: 0=balanced, 1=unbalanced
base_patterns = {
    0: np.array([0.50, 0.50, 0.50, 0.50]),
    1: np.array([0.90, 0.10, 0.80, 0.20]),
}

n_per_class = 600   
base_noise = 0.08   
aug_noise = 0.04    

def generate_data(base_patterns, n_per_class=600, noise_std=0.08):
    X_list, y_list = [], []
    for label, base in base_patterns.items():
        for _ in range(n_per_class):
            sample = base + np.random.normal(0, noise_std, size=4)
            sample = np.clip(sample, 0, 1)
            X_list.append(sample)
            y_list.append(label)
    return np.array(X_list), np.array(y_list)

def augment_data(X, y, noise_std=0.04, repeats=2, flip_prob=0.35):
    X_parts = [X]
    y_parts = [y]

    for _ in range(repeats):
        # 1) jitter/noise augmentation
        X_noise = np.clip(X + np.random.normal(0, noise_std, X.shape), 0, 1)
        X_parts.append(X_noise)
        y_parts.append(y)

        # 2) left-right flip augmentation: [fl, fr, bl, br] -> [fr, fl, br, bl]
        X_flip = X.copy()
        mask = np.random.rand(len(X)) < flip_prob
        X_flip[mask] = X_flip[mask][:, [1, 0, 3, 2]]
        X_parts.append(X_flip)
        y_parts.append(y)

    X_all = np.vstack(X_parts)
    y_all = np.concatenate(y_parts)
    return X_all, y_all

X_raw, y_raw = generate_data(base_patterns, n_per_class=n_per_class, noise_std=base_noise)
X, y = augment_data(X_raw, y_raw, noise_std=aug_noise, repeats=2, flip_prob=0.35)

print('Raw samples:', len(X_raw))
print('After augmentation:', len(X))

Raw samples: 1200
After augmentation: 6000


In [9]:
# 2. train_test_split + Model Training + accuracy
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_blc = RandomForestClassifier(n_estimators=200, random_state=42)
model_blc.fit(X_train, y_train)

train_pred = model_blc.predict(X_train)
test_pred = model_blc.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

print(f'Train Accuracy: {train_acc:.4f}')
print(f'Test  Accuracy: {test_acc:.4f}')

Train Accuracy: 1.0000
Test  Accuracy: 1.0000


In [ ]:
def pick_threshold(model, X_val, y_val, step=0.01):
    p_bad = model.predict_proba(X_val)[:, 1]

    y05 = (p_bad >= 0.5).astype(int)
    print("threshold=0.5",
          "acc", round(accuracy_score(y_val, y05), 3),
          "prec", round(precision_score(y_val, y05, zero_division=0), 3),
          "rec", round(recall_score(y_val, y05, zero_division=0), 3),
          "f1", round(f1_score(y_val, y05, zero_division=0), 3))

    best_thr, best_f1 = 0.5, -1
    for thr in np.arange(0.05, 0.96, step):
        y_hat = (p_bad >= thr).astype(int)
        f1 = f1_score(y_val, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)

    print("best_thr", best_thr, "best_f1", round(best_f1, 3))
    return best_thr

In [15]:
thr = pick_threshold(model_blc, X_test, y_test)
print(thr)

threshold=0.5 acc 1.0 prec 1.0 rec 1.0 f1 1.0
best_thr 0.45000000000000007 best_f1 1.0
0.45000000000000007


In [4]:
# 3. export
joblib.dump(model_blc, 'model_blc.pkl')
print("Base model 'model_blc.pkl' has been saved.")

Base model 'model_blc.pkl' has been saved.
